In [11]:
# ============================================
# 1. Import Libraries
# ============================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [12]:
# ============================================
# 2. Load Data
# ============================================

male_file = "male_file.xlsx"
female_file = "female_file.xlsx"

male = pd.read_excel(male_file)
female = pd.read_excel(female_file)

data = pd.concat([male, female], ignore_index=True)

data.columns = data.columns.str.strip()

In [13]:
# ============================================
# 3. Encode Sex
# ============================================

data["Sex"] = data["Sex"].map({"M":0, "F":1})

In [14]:
# ============================================
# 4. Prepare Leaf-Level Features
# ============================================

leaf_features = data.drop(columns=["Sex","Tree_ID"])

In [15]:
# ============================================
# 5. Build Tree-Level Dataset
# ============================================

tree_mean = data.groupby("Tree_ID").mean(numeric_only=True)
tree_std = data.groupby("Tree_ID").std(numeric_only=True)

tree_dataset = tree_mean.join(tree_std, rsuffix="_std")

tree_sex = data.groupby("Tree_ID")["Sex"].first()

tree_dataset["Sex"] = tree_sex


print("Tree-level dataset shape:", tree_dataset.shape)

Tree-level dataset shape: (172, 24)


In [16]:
# ============================================
# 6. Prepare Training Data
# ============================================

X = tree_dataset.drop(columns=["Sex"])
y = tree_dataset["Sex"]

In [17]:
# ============================================
# 7. Train/Test Split
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [18]:
# ============================================
# 8. Define Models
# ============================================

models = {

    "RandomForest": RandomForestClassifier(
        n_estimators=400,
        random_state=42
    ),

    "SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=42
    ),

    "LogisticRegression": LogisticRegression(
        max_iter=2000
    ),

    "GradientBoosting": GradientBoostingClassifier(
        random_state=42
    )
}

In [19]:
# ============================================
# 9. Train Models
# ============================================

results = []

for model_name, model in models.items():

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    results.append({
        "Model": model_name,
        "Accuracy": acc,
        "F1-score": f1,
        "Precision": precision,
        "Recall": recall
    })

In [20]:
# ============================================
# 10. Display Results
# ============================================

results_df = pd.DataFrame(results)

print("\nTree-Level Model Comparison:\n")
print(results_df.sort_values(by="Accuracy", ascending=False))


Tree-Level Model Comparison:

                Model  Accuracy  F1-score  Precision    Recall
0        RandomForest  0.596154  0.511628   0.647059  0.423077
1                 SVM  0.576923  0.388889   0.700000  0.269231
3    GradientBoosting  0.557692  0.566038   0.555556  0.576923
2  LogisticRegression  0.519231  0.528302   0.518519  0.538462
